# Avaliação Prática 1 — CIFAR-10 · executor Kaggle

Variante do [`colab.ipynb`](colab.ipynb). O Kaggle tem cota de GPU **independente** da do
Colab (30 h/semana), o que o torna a rota de fuga quando o Colab esgota o limite de uso —
e o CIFAR-10 baixa sozinho pelo Keras, então não há upload de dados a fazer.

**Antes de rodar**, no painel direito (`⋮` → Settings):

1. `Accelerator` → **GPU T4 x2** (ou P100)
2. `Internet` → **On** — sem isso o `git clone` e o download dos pesos ImageNet falham

**Persistência:** o Kaggle mantém `/kaggle/working` entre execuções da *mesma* sessão, mas
não entre sessões. Baixe o zip (última célula) antes de encerrar — cada JSON custou GPU.

In [ ]:
# 1. Clonar (ou atualizar) o repositório e instalar as dependências.
import pathlib, subprocess

REPO = pathlib.Path('/kaggle/working/machine-learning-fundamentals')
ACTIVITY = REPO / 'activities' / 'avaliacao-pratica-1'

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/fsd-dantas/machine-learning-fundamentals.git',
                    str(REPO)], check=True)

assert ACTIVITY.is_dir(), f'{ACTIVITY} não existe no repositório publicado.'
%cd {ACTIVITY}
!pip install -q -r requirements.txt

import tensorflow as tf, torch, transformers
print('TensorFlow ', tf.__version__, '| GPU:', bool(tf.config.list_physical_devices('GPU')))
print('PyTorch    ', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('transformers', transformers.__version__, '(precisa ser 4.x — a 5.x quebra o ViT)')
assert torch.cuda.is_available(), 'Sem GPU: Settings → Accelerator → GPU T4 x2'
assert transformers.__version__ < '5', "rode: !pip install -q 'transformers>=4.40,<5'"

In [ ]:
# 2. RESTAURAR resultados anteriores (opcional).
#    Se você já rodou parte do experimento em outra sessão/plataforma e baixou o zip,
#    anexe-o como dataset (+ Add Input → Upload) e esta célula recupera os JSONs —
#    o status.py então pedirá apenas o que falta, e nenhuma GPU é gasta duas vezes.
import pathlib, shutil, zipfile

RESULTS = pathlib.Path('results'); RESULTS.mkdir(exist_ok=True)
recuperados = 0

for entrada in pathlib.Path('/kaggle/input').rglob('*'):
    if entrada.suffix == '.json' and entrada.name.startswith(('s1_', 's2_', 's3_', 's4_', 's5_')):
        shutil.copy(entrada, RESULTS / entrada.name); recuperados += 1
    elif entrada.suffix == '.zip':
        with zipfile.ZipFile(entrada) as z:
            for nome in z.namelist():
                if nome.endswith('.json') and '/results/' in nome:
                    (RESULTS / pathlib.Path(nome).name).write_bytes(z.read(nome))
                    recuperados += 1

print(f'{recuperados} artefatos recuperados de /kaggle/input')
print(f'{len(list(RESULTS.glob("*.json")))} JSONs em results/')

In [ ]:
# 3. O QUE FALTA — inventário e comandos exatos.
#    run_all.py NÃO pula o que já existe: reexecutar uma etapa retreina tudo nela.
#    Rode apenas os comandos que este script imprimir.
!python status.py

In [ ]:
# 4. CORE — as 5 estratégias na seed primária (~35 min).
#    Pule se o status.py já indicou 5/5.
!python run_all.py --stage core
!python report.py

In [ ]:
# 5. ABLAÇÕES BARATAS — questões 2(a) e 4(a) (~35 min).
!python s2_feature_extraction.py --ablation
!python s3_finetuning.py --ablation-head --seeds 42 7
!python report.py

In [ ]:
# 6. ABLAÇÕES CARAS — questões 4(c) e 4(b) (~2h40 com orçamento de épocas reduzido).
#    As épocas caem de 15+12 para 8+6, IDÊNTICAS em todos os braços: a validade interna
#    da comparação se mantém (a pergunta é qual braço vence qual), mas as acurácias
#    absolutas das ablações não são comparáveis à tabela principal. Declarado no relatório.
!python s4_augmentation.py --ablation-policy    --seeds 42 7 --epochs-frozen 8 --epochs-finetune 6
!python report.py
!python s4_augmentation.py --ablation-optimizer --seeds 42 7 --epochs-frozen 8 --epochs-finetune 6
!python report.py

In [ ]:
# 7. OPCIONAL — dispersão da tabela principal e o setup do notebook da aula.
!python run_all.py --stage seeds
!python run_all.py --stage baseline
!python report.py

In [ ]:
# 8. BAIXAR — rode isto CEDO e com frequência. Cada JSON custou GPU, e a sessão
#    do Kaggle não sobrevive a um fechamento de aba.
import shutil, pathlib

OUT = pathlib.Path('/kaggle/working/ap1-results')
if OUT.exists():
    shutil.rmtree(OUT)
shutil.copytree('results', OUT / 'results')

(OUT / 'img').mkdir()
for png in pathlib.Path('../../assets/img').glob('avaliacao-pratica-1-*.png'):
    shutil.copy(png, OUT / 'img' / png.name)

shutil.make_archive('/kaggle/working/ap1-results', 'zip', OUT)
n = len(list((OUT / 'results').glob('*.json')))
print(f'/kaggle/working/ap1-results.zip · {n} artefatos')
print('baixe pela aba Output (painel direito)')